In [10]:
import json
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# ---------------------------------------------------------
# 1. Load the FAQ Dataset
# ---------------------------------------------------------
with open('faqs.json', 'r') as f:
    raw_faqs = json.load(f)

# ---------------------------------------------------------
# 2. Perform Document-Level Chunking (1 FAQ = 1 Document)
# ---------------------------------------------------------
documents = []
for item in raw_faqs:
    # Combine Category, Question, and Answer into one unified text chunk
    formatted_content = (
        f"Category: {item['category']}\n"
        f"Question: {item['question']}\n"
        f"Answer: {item['answer']}"
    )
    
    # Create a LangChain Document object with metadata
    doc = Document(
        page_content=formatted_content,
        metadata={
            "faq_id": item["id"],
            "category": item["category"],
            "question": item["question"]
        }
    )
    documents.append(doc)

print(f"Successfully created {len(documents)} document chunks.")

# ---------------------------------------------------------
# 3. Initialize Free Embedding Model (Hugging Face)
# ---------------------------------------------------------
# Using all-MiniLM-L6-v2 (Lightweight, fast, and great for sentence similarity)
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# ---------------------------------------------------------
# 4. Store Chunks in ChromaDB Vector Database
# ---------------------------------------------------------
vector_store = Chroma.from_documents(
    documents=documents,
    embedding=embedding_model,
    persist_directory="./chroma_db"  # Saves database locally in Colab
)

print("Vector Database created and indexed successfully!")

# ---------------------------------------------------------


Successfully created 200 document chunks.


Loading weights: 100%|█████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 2532.12it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Vector Database created and indexed successfully!


In [14]:
import json
import os
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from langchain_community.chat_message_histories import SQLChatMessageHistory
from langchain_community.llms import Ollama
from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnablePassthrough
from pydantic import BaseModel

# 1. Initialize local LLM & Retriever
llm = Ollama(model="phi3:mini")
retriever = vector_store.as_retriever(search_kwargs={"k": 2})

# 2. Decompose Multi-Question Inputs Prompt
decompose_system_prompt = """You are an query analyzer for a college chatbot. 
Analyze the user's input alongside the chat history. 
If the user asks multiple questions, split them into standalone questions.
If the question relies on context (e.g. "what is her email?"), rewrite it using the chat history to be standalone.

Output ONLY a JSON array of strings containing the individual questions. No explanations or extra text.

Example Output:
["Who is the HOD of AI and Machine Learning?", "What is the fee for B.E. CSE?"]
"""

decompose_prompt = ChatPromptTemplate.from_messages([
    ("system", decompose_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

decompose_chain = decompose_prompt | llm | StrOutputParser()


def process_and_retrieve_docs(input_dict):
  raw_user_input = input_dict["input"]
  chat_history = input_dict.get("chat_history", [])

  try:
    decomposition_output = decompose_chain.invoke(
        {"input": raw_user_input, "chat_history": chat_history}
    )
    questions = json.loads(decomposition_output.strip())
    if not isinstance(questions, list):
      questions = [raw_user_input]
  except Exception:
    questions = [raw_user_input]

  all_docs = []
  seen_contents = set()

  for q in questions:
    docs = retriever.invoke(q)
    for doc in docs:
      if doc.page_content not in seen_contents:
        seen_contents.add(doc.page_content)
        all_docs.append(doc)

  return "\n\n".join(doc.page_content for doc in all_docs)


# 3. Final Multi-Answer Generation Prompt
qa_system_prompt = """You are a helpful College FAQ Chatbot. 
Answer ALL parts of the user's query clearly using ONLY the provided context below. 
Address each asked question distinctly using bullet points or concise paragraphs.
If the context doesn't contain information for a specific sub-question, explicitly state that you don't have information on that part.

Context:
{context}"""

qa_prompt = ChatPromptTemplate.from_messages([
    ("system", qa_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

rag_chain = (
    RunnablePassthrough.assign(context=process_and_retrieve_docs)
    | qa_prompt
    | llm
    | StrOutputParser()
)


# 4. Core Core Response Helper Function (Used by API)
def get_chat_response(session_id: str, user_message: str) -> str:
  history = SQLChatMessageHistory(
      session_id=session_id, connection_string="sqlite:///chat_history.db"
  )

  response = rag_chain.invoke(
      {"input": user_message, "chat_history": history.messages}
  )

  history.add_user_message(user_message)
  history.add_ai_message(response)

  return response


# 5. FastAPI App Setup
app = FastAPI(title="College FAQ Chatbot API")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)


class ChatRequest(BaseModel):
  session_id: str
  message: str


@app.post("/api/chat")
def chat_endpoint(request: ChatRequest):
  reply = get_chat_response(request.session_id, request.message)
  return {"status": "success", "response": reply}

Ask the Query Who is HOD of AI and ML department, and what is the fee for B.E. Computer Science?


User: Who is HOD of AI and ML department, and what is the fee for B.E. Computer Science?

Bot:
- Question: Who is the Head of Department (HOD) for Artificial Intelligence & Machine Learning at TJIT College?
  
    Answer: Dr. Asha S. Manek heads the AI & ML department.
    
- Question: What is the fee structure for B.E in Computer Science Engineering at TJIT College?
  
    Answer: The total course fee for a Bachelor of Engineering (B.E.) degree in Computer Science and Engineering (CSE) at TJIT college is approximately INR 8.6 Lakhs, covering the span over four years.

